In [60]:
import re
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
MANIFEST_DIR = Path("/glade/derecho/scratch/bbuchovecky/cmip_intake_esgf_fetch/manifests")

# Generate esgf_cache_catalog.csv with:
#  > python /glade/u/home/bbuchovecky/pyth/cmip-intake-esgf-fetch/scripts/catalog_esgf_cache.py \
#  > --cache-dir /glade/derecho/scratch/bbuchovecky/cmip_intake_esgf_fetch/esgf_cache` \
#  > --output /glade/derecho/scratch/bbuchovecky/cmip_intake_esgf_fetch/esgf_cache/manifests/esgf_cache_catalog.csv \
#  > --no-open
catalog = pd.read_csv(MANIFEST_DIR / "esgf_cache_catalog.csv")

In [5]:
catalog.columns

Index(['status', 'error', 'path', 'relative_path', 'filename', 'size_bytes',
       'modified_utc', 'mip_era', 'project_id', 'activity_id',
       'institution_id', 'source_id', 'model_id', 'experiment_id', 'member_id',
       'ensemble', 'table_id', 'frequency', 'realm', 'variable_id',
       'grid_label', 'time_range', 'version', 'tracking_id',
       'nominal_resolution', 'data_vars', 'dims', 'time_name', 'n_time',
       'time_start', 'time_stop', 'lat_name', 'lon_name', 'y_name', 'x_name',
       'n_lat', 'n_lon', 'approx_dlat', 'approx_dlon', 'has_bounds',
       'has_areacella', 'has_sftlf'],
      dtype='str')

In [6]:
catalog["source_id"].unique()

<ArrowStringArray>
[          'TaiESM1',     'AWI-CM-1-1-MR',    'AWI-ESM-1-1-LR',
   'AWI-ESM-1-REcoM',       'BCC-CSM2-MR',          'BCC-ESM1',
       'CAMS-CSM1-0',        'CAS-ESM2-0',       'FGOALS-f3-L',
         'FGOALS-g3',          'IITM-ESM',           'CanESM5',
         'CanESM5-1',     'CanESM5-CanOE',      'CMCC-CM2-HR4',
      'CMCC-CM2-SR5',         'CMCC-ESM2',        'CNRM-CM6-1',
     'CNRM-CM6-1-HR',       'CNRM-ESM2-1',     'ACCESS-ESM1-5',
        'ACCESS-CM2',          'E3SM-1-0',          'E3SM-1-1',
      'E3SM-1-1-ECA',          'E3SM-2-0',    'E3SM-2-0-NARRM',
          'E3SM-2-1',         'EC-Earth3', 'EC-Earth3-AerChem',
      'EC-Earth3-CC',   'EC-Earth3-ESM-1',      'EC-Earth3-HR',
      'EC-Earth3-LR',     'EC-Earth3-Veg',  'EC-Earth3-Veg-LR',
       'FIO-ESM-2-0',   'MPI-ESM-1-2-HAM',         'INM-CM4-8',
         'INM-CM5-0',   'IPSL-CM5A2-INCA',      'IPSL-CM6A-LR',
 'IPSL-CM6A-LR-INCA',     'IPSL-CM6A-MR1',         'KIOST-ESM',
        'MIROC-ES2H',

In [8]:
catalog["variable_id"].unique()

<ArrowStringArray>
[ 'areacella',      'sftlf',    'evspsbl', 'evspsblsoi', 'evspsblveg',
        'gpp',        'lai',       'tran']
Length: 8, dtype: str

In [282]:
# query = (catalog["experiment_id"] == "historical")
query = True

source_with_area = set(catalog.loc[(catalog["variable_id"] == "areacella") & query].drop_duplicates(subset=['source_id'], keep='first')["source_id"].tolist())
source_with_sftlf = set(catalog.loc[(catalog["variable_id"] == "sftlf") & query].drop_duplicates(subset=['source_id'], keep='first')["source_id"].tolist())

source_with_landgrid = source_with_area & source_with_sftlf
print(f"number of models with areacella and sftlf: {len(source_with_landgrid)}")

number of models with areacella and sftlf: 54


In [323]:
# variables = ["evspsbl", "evspsblsoi", "evspsblveg", "lai", "tran", "gpp"]
variables = ["evspsbl"]
experiment_id = "historical"

avail_member_id = {}
avail_variables = {}
avail_source_id = []

for sid in sorted(source_with_landgrid):
    print(f"=== {sid} ===")
    avail_member_id[sid] = {}
    avail_variables[sid] = []

    num_avail_vars = 0
    avail_mid = set(catalog.member_id.unique())  # initialize member_ids available for all variables

    for var in variables:
        query = (catalog["experiment_id"] == experiment_id) & (catalog["source_id"] == sid) & (catalog["variable_id"] == var)
        subset = catalog.loc[query]

        if len(subset) > 0:
            print(f"  ✅ {var:11}")

            unique_member_id = subset.member_id.unique()
            avail_mid = avail_mid & set(unique_member_id)
            avail_variables[sid].append(var) 
            num_avail_vars += 1
        
        else:
            print(f"  ❌ {var}")

    avail_member_id[sid] = avail_mid
    if len(avail_mid) == 0:
        print("  ⚠️ no shared member_id")
    elif num_avail_vars == len(variables):
        avail_source_id.append(sid)

=== ACCESS-CM2 ===
  ✅ evspsbl    
=== ACCESS-ESM1-5 ===
  ✅ evspsbl    
=== AWI-CM-1-1-MR ===
  ✅ evspsbl    
=== AWI-ESM-1-1-LR ===
  ✅ evspsbl    
=== AWI-ESM-1-REcoM ===
  ✅ evspsbl    
=== BCC-ESM1 ===
  ✅ evspsbl    
=== CAMS-CSM1-0 ===
  ✅ evspsbl    
=== CMCC-CM2-SR5 ===
  ✅ evspsbl    
=== CMCC-ESM2 ===
  ✅ evspsbl    
=== CNRM-CM6-1 ===
  ✅ evspsbl    
=== CNRM-CM6-1-HR ===
  ✅ evspsbl    
=== CNRM-ESM2-1 ===
  ✅ evspsbl    
=== CanESM5 ===
  ✅ evspsbl    
=== CanESM5-1 ===
  ✅ evspsbl    
=== CanESM5-CanOE ===
  ✅ evspsbl    
=== E3SM-1-0 ===
  ✅ evspsbl    
=== E3SM-1-1 ===
  ✅ evspsbl    
=== E3SM-1-1-ECA ===
  ✅ evspsbl    
=== E3SM-2-0 ===
  ✅ evspsbl    
=== E3SM-2-0-NARRM ===
  ✅ evspsbl    
=== E3SM-2-1 ===
  ✅ evspsbl    
=== EC-Earth3 ===
  ✅ evspsbl    
=== EC-Earth3-AerChem ===
  ✅ evspsbl    
=== EC-Earth3-ESM-1 ===
  ✅ evspsbl    
=== EC-Earth3-HR ===
  ✅ evspsbl    
=== EC-Earth3-Veg ===
  ✅ evspsbl    
=== EC-Earth3-Veg-LR ===
  ✅ evspsbl    
=== FGOALS-g3 ===

In [324]:
MEMBER_ID_RE = re.compile(r"r(?P<r>\d+)i(?P<i>\d+)p(?P<p>\d+)f(?P<f>\d+)")

def sort_member_ids(mids):
    return sorted(mids, key=lambda mid: tuple(int(x) for x in MEMBER_ID_RE.match(mid).groups()))

def group_member_ids_by_ipf(mids):
    grouped = {}

    for mid in sort_member_ids(mids):
        match = MEMBER_ID_RE.match(mid)
        if match is None:
            continue

        key = tuple(int(match.group(axis)) for axis in ("i", "p", "f"))
        grouped.setdefault(key, []).append(mid)

    return grouped

member_ids_by_source = {}
member_ids_max_r = {}
top_member_id = {}
sort_avail_member_id = {}

print("n_r is the number of groups of members with constant i,p,f and different r (i.e., number of unique initial condition ensembles)")
print("max_r is the number of members in the i,p,f group with the greatest number of members - this is one selected in 'top_member_id'")
print(f"complete with all variables: {variables}")
print(f"\n{'source_id':17}  {'top_member_id':12}  {'n_member_id':11}  {'n_r':3}  {'max_r':6}  {'complete?'}")
print("-"*69)

for sid, mids in avail_member_id.items():
    if len(mids) > 0:
        sort_avail_mid = sort_member_ids(mids)
        sort_avail_member_id[sid] = sort_avail_mid
        member_ids_by_source[sid] = group_member_ids_by_ipf(mids)

        top_mid = sort_avail_mid[0]
        top_member_id[sid] = top_mid

        nr = len(member_ids_by_source[sid].keys())
        max_r = -np.inf
        max_r_ipf = ""
        for i, s in member_ids_by_source[sid].items():
            if len(s) > max_r:
                max_r = len(s)
                max_r_ipf = s
        member_ids_max_r[sid] = max_r_ipf

        complete_tag = f"❌  {len(avail_variables[sid])}/{len(variables)}   {avail_variables[sid]}"
        if sid in avail_source_id:
            complete_tag = "✅"
        print(f"{sid:17}  {top_mid:13}  {len(avail_member_id[sid]):<11}  {nr:<3}  {max_r:<6}  {complete_tag}")

n_r is the number of groups of members with constant i,p,f and different r (i.e., initial condition ensembles)
max_r is the number of members in the constant i,p,f group with the greatest number of members
complete with all variables: ['evspsbl']

source_id          top_member_id  n_member_id  n_r  max_r   complete?
---------------------------------------------------------------------
ACCESS-CM2         r1i1p1f1       10           1    10      ✅
ACCESS-ESM1-5      r1i1p1f1       40           1    40      ✅
AWI-CM-1-1-MR      r1i1p1f1       5            1    5       ✅
AWI-ESM-1-1-LR     r1i1p1f1       1            1    1       ✅
AWI-ESM-1-REcoM    r1i1p1f1       1            1    1       ✅
BCC-ESM1           r1i1p1f1       3            1    3       ✅
CAMS-CSM1-0        r1i1p1f1       3            2    2       ✅
CMCC-CM2-SR5       r1i1p1f1       11           2    10      ✅
CMCC-ESM2          r1i1p1f1       1            1    1       ✅
CNRM-CM6-1         r1i1p1f2       30           1    30

In [327]:
TIME_SLICE = slice("1950-01", "2014-12")
LOAD_ALL_REALS = True

data_dict = {}
for sid in sorted(avail_source_id):
    print(f"=== {sid} ===")
    
    data_dict[sid] = {}

    if not LOAD_ALL_REALS:
        # Only load the top (single) member_id
        mids = [top_member_id[sid]]
    else:
        # Load all realizations
        mids = member_ids_max_r[sid]

    for var in variables:
        print(f"  ✅ {var:11}  n={len(mids):3}", end="   ")
        das = []

        for mid in mids:
            query = (catalog["experiment_id"] == experiment_id) & (catalog["source_id"] == sid) & (catalog["variable_id"] == var) & (catalog["member_id"] == mid)
            subset = catalog.loc[query]
            row = subset.iloc[0]
            
            if len(subset.loc[query]) > 1:
                da_path = "_".join(row["path"].split("_")[:-1])+"*.nc"
            else:
                da_path = row["path"]

            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                da = xr.open_mfdataset(da_path)[var].sel(time=TIME_SLICE)
                print(mid, end=" ")

            das.append(da)
        
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            data_dict[sid][var] = xr.concat(das, dim="member").assign_coords(member=np.arange(len(mids)), member_id=("member", mids))
        
        print()

=== ACCESS-CM2 ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 r6i1p1f1 r7i1p1f1 r8i1p1f1 r9i1p1f1 r10i1p1f1 
=== ACCESS-ESM1-5 ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 r6i1p1f1 r7i1p1f1 r8i1p1f1 r9i1p1f1 r10i1p1f1 r11i1p1f1 r12i1p1f1 r13i1p1f1 r14i1p1f1 r15i1p1f1 r16i1p1f1 r17i1p1f1 r18i1p1f1 r19i1p1f1 r20i1p1f1 r21i1p1f1 r22i1p1f1 r23i1p1f1 r24i1p1f1 r25i1p1f1 r26i1p1f1 r27i1p1f1 r28i1p1f1 r29i1p1f1 r30i1p1f1 r31i1p1f1 r32i1p1f1 r33i1p1f1 r34i1p1f1 r35i1p1f1 r36i1p1f1 r37i1p1f1 r38i1p1f1 r39i1p1f1 r40i1p1f1 
=== AWI-CM-1-1-MR ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 
=== AWI-ESM-1-1-LR ===
  ✅ evspsbl       r1i1p1f1 
=== AWI-ESM-1-REcoM ===
  ✅ evspsbl       r1i1p1f1 
=== BCC-ESM1 ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 
=== CAMS-CSM1-0 ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 
=== CMCC-CM2-SR5 ===
  ✅ evspsbl       r2i1p2f1 r3i1p2f1 r4i1p2f1 r5i1p2f1 r6i1p2f1 r7i1p2f1 r8i1p2f1 r9i1p2f1 r10i1p2f1 r11i1p2f1

/glade/derecho/scratch/bbuchovecky/tmp/ipykernel_36386/2504939030.py:31: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  data_dict[sid][var] = xr.concat(das, dim="member").assign_coords(member=np.arange(len(mids)), member_id=("member", mids))



=== E3SM-1-1 ===
  ✅ evspsbl       r1i1p1f1 
=== E3SM-1-1-ECA ===
  ✅ evspsbl       r1i1p1f1 
=== E3SM-2-0 ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 r6i1p1f1 r7i1p1f1 r8i1p1f1 r9i1p1f1 r10i1p1f1 r11i1p1f1 r12i1p1f1 r13i1p1f1 r14i1p1f1 r15i1p1f1 r16i1p1f1 r17i1p1f1 r18i1p1f1 r19i1p1f1 r20i1p1f1 r21i1p1f1 
=== E3SM-2-0-NARRM ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 
=== E3SM-2-1 ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 
=== EC-Earth3 ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 r6i1p1f1 r7i1p1f1 r9i1p1f1 r10i1p1f1 r11i1p1f1 r12i1p1f1 r13i1p1f1 r14i1p1f1 r15i1p1f1 r16i1p1f1 r17i1p1f1 r18i1p1f1 r19i1p1f1 r20i1p1f1 r21i1p1f1 r22i1p1f1 r23i1p1f1 r24i1p1f1 r25i1p1f1 r101i1p1f1 r102i1p1f1 r103i1p1f1 r104i1p1f1 r105i1p1f1 r106i1p1f1 r107i1p1f1 r108i1p1f1 r109i1p1f1 r110i1p1f1 r111i1p1f1 r112i1p1f1 r113i1p1f1 r114i1p1f1 r115i1p1f1 r116i1p1f1 r117i1p1f1 r118i1p1f1 r119i1p1f1 r120i1p1f1 r121i1p1f1 r

/glade/derecho/scratch/bbuchovecky/tmp/ipykernel_36386/2504939030.py:31: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  data_dict[sid][var] = xr.concat(das, dim="member").assign_coords(member=np.arange(len(mids)), member_id=("member", mids))
/glade/derecho/scratch/bbuchovecky/tmp/ipykernel_36386/2504939030.py:31: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'lat' ('lat',) The recommendation is to set join explicitly for this case.
  da


=== EC-Earth3-AerChem ===
  ✅ evspsbl       r1i1p1f1 r3i1p1f1 r4i1p1f1 
=== EC-Earth3-ESM-1 ===
  ✅ evspsbl       

...

r5i1p1f1 
=== EC-Earth3-HR ===
  ✅ evspsbl       r1i1p1f1 
=== EC-Earth3-Veg ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 r6i1p1f1 r10i1p1f1 r11i1p1f1 r12i1p1f1 r13i1p1f1 r14i1p1f1 

/glade/derecho/scratch/bbuchovecky/tmp/ipykernel_36386/2504939030.py:31: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'lat' ('lat',) The recommendation is to set join explicitly for this case.
  data_dict[sid][var] = xr.concat(das, dim="member").assign_coords(member=np.arange(len(mids)), member_id=("member", mids))



=== EC-Earth3-Veg-LR ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 
=== FGOALS-g3 ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 r6i1p1f1 
=== GFDL-ESM4 ===
  ✅ evspsbl       r1i1p1f1 
=== GISS-E2-1-G ===
  ✅ evspsbl       r1i1p1f2 r2i1p1f2 r3i1p1f2 r4i1p1f2 r5i1p1f2 r6i1p1f2 r7i1p1f2 r8i1p1f2 r9i1p1f2 r10i1p1f2 r11i1p1f2 r12i1p1f2 r13i1p1f2 r14i1p1f2 r15i1p1f2 r16i1p1f2 r17i1p1f2 r18i1p1f2 r19i1p1f2 r20i1p1f2 r201i1p1f2 r202i1p1f2 r203i1p1f2 r204i1p1f2 r205i1p1f2 r206i1p1f2 r207i1p1f2 r208i1p1f2 r209i1p1f2 r210i1p1f2 r301i1p1f2 r302i1p1f2 r303i1p1f2 r304i1p1f2 r305i1p1f2 r306i1p1f2 r307i1p1f2 r308i1p1f2 r309i1p1f2 r310i1p1f2 
=== GISS-E2-1-G-CC ===
  ✅ evspsbl       

/glade/derecho/scratch/bbuchovecky/tmp/ipykernel_36386/2504939030.py:31: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  data_dict[sid][var] = xr.concat(das, dim="member").assign_coords(member=np.arange(len(mids)), member_id=("member", mids))


r1i1p1f1 
=== GISS-E2-1-H ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 r6i1p1f1 r7i1p1f1 r8i1p1f1 r9i1p1f1 r10i1p1f1 
=== GISS-E2-2-G ===
  ✅ evspsbl       r1i1p1f1 r2i1p1f1 r3i1p1f1 r4i1p1f1 r5i1p1f1 r6i1p1f1 
=== GISS-E3-G ===
  ✅ evspsbl       r1i1p101f1 
=== HadGEM3-GC31-LL ===
  ✅ evspsbl       r1i1p1f3 r2i1p1f3 r3i1p1f3 r4i1p1f3 r5i1p1f3 r11i1p1f3 r12i1p1f3 r13i1p1f3 r14i1p1f3 r15i1p1f3 r16i1p1f3 r17i1p1f3 r18i1p1f3 r19i1p1f3 r20i1p1f3 r21i1p1f3 r22i1p1f3 r23i1p1f3 r24i1p1f3 r25i1p1f3 r26i1p1f3 r27i1p1f3 r28i1p1f3 r29i1p1f3 r30i1p1f3 r31i1p1f3 r32i1p1f3 r33i1p1f3 r34i1p1f3 r35i1p1f3 r36i1p1f3 r37i1p1f3 r38i1p1f3 r39i1p1f3 r40i1p1f3 r41i1p1f3 r42i1p1f3 r43i1p1f3 r44i1p1f3 r45i1p1f3 r46i1p1f3 r47i1p1f3 r48i1p1f3 r49i1p1f3 r50i1p1f3 r51i1p1f3 r52i1p1f3 r53i1p1f3 r54i1p1f3 r55i1p1f3 r56i1p1f3 r57i1p1f3 r58i1p1f3 r59i1p1f3 r60i1p1f3 
=== HadGEM3-GC31-MM ===
  ✅ evspsbl       r1i1p1f3 r2i1p1f3 r3i1p1f3 r4i1p1f3 
=== ICON-ESM-LR ===
  ✅ evspsbl       r1i1p1f1 r2i1p

In [330]:
data_dict["MRI-ESM2-0"]["evspsbl"]

<xarray.DataArray 'evspsbl' (member: 10, time: 780, lat: 160, lon: 320)> Size: 2GB
dask.array<concatenate, shape=(10, 780, 160, 320), dtype=float32, chunksize=(1, 1, 160, 320), chunktype=numpy.ndarray>
Coordinates:
  * member     (member) int64 80B 0 1 2 3 4 5 6 7 8 9
    member_id  (member) <U9 360B 'r1i1p1f1' 'r2i1p1f1' ... 'r10i1p1f1'
  * time       (time) datetime64[ns] 6kB 1950-01-16T12:00:00 ... 2014-12-16T1...
  * lat        (lat) float64 1kB -89.14 -88.03 -86.91 ... 86.91 88.03 89.14
  * lon        (lon) float64 3kB 0.0 1.125 2.25 3.375 ... 356.6 357.8 358.9
Attributes:
    standard_name:  water_evapotranspiration_flux
    long_name:      Evaporation Including Sublimation and Transpiration
    comment:        Evaporation at surface (also known as evapotranspiration)...
    units:          kg m-2 s-1
    original_name:  EVSPS
    cell_methods:   area: time: mean
    cell_measures:  area: areacella
    history:        2019-02-20T02:38:09Z altered by CMOR: replaced missing va...